# Twitter Sentiment Analysis🤖




### This project presents an AI-powered Twitter Sentiment Analysis system developed using Natural Language Processing and Machine Learning techniques.
### The Sentiment140 dataset is utilized to classify tweets as positive or negative based on their textual content.
### Comprehensive text preprocessing, TF-IDF feature extraction, and multiple machine learning algorithms are implemented and evaluated.
### The final system accurately predicts sentiment from new tweets and demonstrates the practical application of AI in social media analytics.


### STEP 1 : IMPORT LIBRARIES
#### Import all required libraries for data processing, visualization, NLP, and machine learning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re

import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

### STEP 2 : DOWNLOAD NLTK RESOURCES
#### Download necessary NLP resources such as stopwords and lemmatization data.

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

### STEP 3 : LOAD DATASET
#### Load the Sentiment140 dataset into a DataFrame for analysis.


In [ ]:
columns = [
    'target',
    'id',
    'date',
    'flag',
    'user',
    'text'
]

df = pd.read_csv(
    "/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv",
    encoding='latin-1',
    names=columns
)

df.head()

### STEP 4 : DATA OVERVIEW
#### Understand the dataset structure, size, and data quality.

In [ ]:
print(df.shape)

df.info()

df.isnull().sum()

### STEP 5 : KEEP REQUIRED COLUMNS
#### Retain only the columns needed for sentiment analysis.

In [ ]:
df = df[['target','text']]
df.head()

### STEP 6 : CONVERT LABELS
#### Convert sentiment labels into a binary format for model training.

In [ ]:
df['target'] = df['target'].replace(4,1)

df['target'].value_counts()

### STEP 7 : VISUALIZE SENTIMENT DISTRIBUTION
#### Examine the balance between positive and negative sentiments.

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    x=df['target']
)

plt.title("Sentiment Distribution")
plt.show()

### STEP 8 : TEXT CLEANING FUNCTION
#### Remove unwanted text elements and normalize tweet content.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"\d+", "", text)

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

### STEP 9 : CLEAN TWEETS
#### Apply preprocessing to prepare tweets for feature extraction.

In [ ]:
df['clean_text'] = df['text'].apply(clean_text)

df.head()

### STEP 10 : CHECK SAMPLE TWEETS
#### Verify that text cleaning is working correctly.

In [ ]:
for i in range(5):
    print("Original:")
    print(df['text'][i])

    print("\nCleaned:")
    print(df['clean_text'][i])

    print("="*70)

### STEP 11 : TF-IDF VECTORIZATION
#### Transform text data into numerical features for machine learning.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000
)

X = vectorizer.fit_transform(df['clean_text'])

y = df['target']

### STEP 12 : TRAIN TEST SPLIT
#### Separate data into training and testing sets for evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### STEP 13 : LOGISTIC REGRESSION
#### Train a baseline sentiment classification model.

In [ ]:
lr_model = LogisticRegression()

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_acc = accuracy_score(
    y_test,
    lr_pred
)

print("Logistic Regression Accuracy:", lr_acc)

### STEP 14 : NAIVE BAYES
#### Train a probabilistic model suitable for text classification.

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)

nb_acc = accuracy_score(
    y_test,
    nb_pred
)

print("Naive Bayes Accuracy:", nb_acc)

### STEP 15 : COMPARE MODELS
#### Evaluate and compare the accuracy of all trained models.

In [ ]:
results = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "Naive Bayes"  
    ],

    "Accuracy":[
        lr_acc,
        nb_acc
    ]
})

results

### STEP 16 : ACCURACY COMPARISON GRAPH
#### Visualize model performance for easier comparison.

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x="Model",
    y="Accuracy",
    data=results
)

plt.title("Model Accuracy Comparison")

plt.xticks(rotation=20)

plt.show()

### STEP 17 : CONFUSION MATRIX
#### Analyze correct and incorrect predictions made by the model.

In [ ]:
cm = confusion_matrix(
    y_test,
    lr_pred
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

### STEP 18 : CLASSIFICATION REPORT
#### Measure precision, recall, F1-score, and overall performance.

In [ ]:
print(
    classification_report(
        y_test,
        lr_pred
    )
)

### STEP 19 : SENTIMENT PREDICTION FUNCTION
#### Create a function to predict sentiment for new tweets.

In [ ]:
def predict_sentiment(text):

    cleaned = clean_text(text)

    vectorized = vectorizer.transform(
        [cleaned]
    )

    prediction = lr_model.predict(
        vectorized
    )[0]

    if prediction == 1:
        return "Positive 😊"

    else:
        return "Negative 😔"

### STEP 20 : TEST CUSTOM TWEETS
#### Validate the model using user-defined tweet examples.

In [ ]:
sample1 = "This is the worst day ever.."

sample2 = "I am extremely happy today"

print(
    predict_sentiment(sample1)
)

print(
    predict_sentiment(sample2)
)

### STEP 21 : SAVE MODEL
#### Store the trained model and vectorizer for future use or deployment.

In [ ]:
import pickle

pickle.dump(
    lr_model,
    open("twitter_sentiment_model.pkl","wb")
)

pickle.dump(
    vectorizer,
    open("tfidf_vectorizer.pkl","wb")
)

print("Model Saved Successfully")